# KG Fact-Checking Engine

Given the **KG-2022** train/test files (reified N-Triples over DBpedia
entities/relations), this notebook builds an engine that scores an
arbitrary `(subject, predicate, object)` fact with a **veracity value in
`[0, 1]`** (0 = false, 1 = true), using the knowledge graph itself as the
evidence base.

**Structure:**
1. Parse & analyse the training data
2. Turn the KG into features (structural popularity + known-facts evidence
   + an optional KG-embedding model)
3. Fact-checking engine + benchmarking (baselines, cross-validation,
   ablation study, per-relation breakdown)
4. Interactive single-fact checking + final test-set predictions

> **Data files:** place `KG-2022-train_nt.txt` and `KG-2022-test_nt.txt`
> either next to this notebook, or update `CANDIDATE_DIRS` below.


## Setup


In [1]:
import re
import json
import os
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, brier_score_loss, log_loss)

warnings.filterwarnings('ignore')
pd.set_option('display.width', 140)

# --- locate the data files -------------------------------------------------
CANDIDATE_DIRS = ['.', '../data', './data']

def find_file(fname):
    for d in CANDIDATE_DIRS:
        p = os.path.join(d, fname)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(
        f"Could not find {fname!r}. Place it next to this notebook, or add "
        f"its folder to CANDIDATE_DIRS above."
    )

TRAIN_PATH = find_file('KG-2022-train_nt.txt')
TEST_PATH = find_file('KG-2022-test_nt.txt')
print('train file:', TRAIN_PATH)
print('test file: ', TEST_PATH)


train file: ../data/KG-2022-train_nt.txt
test file:  ../data/KG-2022-test_nt.txt


## 1. Parse the knowledge graph

Each "fact" is a **reified** RDF statement spread across several lines
that all share the same statement-subject URI:

```
<.../dataset/3226691> rdf:type rdf:Statement .
<.../dataset/3226691> swc:hasTruthValue "0.0"^^xsd:float .   # only in train
<.../dataset/3226691> rdf:subject   <dbpedia:David_Lee_(basketball)> .
<.../dataset/3226691> rdf:predicate <dbpedia-ont:team> .
<.../dataset/3226691> rdf:object    <dbpedia:Houston_Rockets> .
```

We parse these into one row per statement.


In [2]:
LINE_RE = re.compile(r'^<([^>]+)>\s+<([^>]+)>\s+(.*?)\s*\.\s*$')

def _clean_object(raw):
    raw = raw.strip()
    if raw.startswith('<') and raw.endswith('>'):
        return raw[1:-1]
    m = re.match(r'^"(.*)"(\^\^<[^>]+>|@[a-zA-Z-]+)?$', raw)
    if m:
        return m.group(1)
    return raw

def parse_statements(path):
    """Return a DataFrame with columns:
    stmt_id, subject, predicate, object, truth_value (NaN if absent)
    """
    records = {}
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            m = LINE_RE.match(line)
            if not m:
                continue
            stmt_uri, pred_uri, obj_raw = m.groups()
            stmt_id = stmt_uri.rsplit('/', 1)[-1]
            rec = records.setdefault(stmt_id, {'stmt_id': stmt_id})
            obj_val = _clean_object(obj_raw)

            if pred_uri.endswith('#type'):
                continue
            elif pred_uri.endswith('hasTruthValue'):
                rec['truth_value'] = float(obj_val)
            elif pred_uri.endswith('#subject'):
                rec['subject'] = obj_val
            elif pred_uri.endswith('#predicate'):
                rec['predicate'] = obj_val
            elif pred_uri.endswith('#object'):
                rec['object'] = obj_val

    df = pd.DataFrame(list(records.values()))
    if 'truth_value' not in df.columns:
        df['truth_value'] = pd.NA

    def local(u):
        if not isinstance(u, str):
            return u
        return u.rsplit('/', 1)[-1]

    df['subject_name'] = df['subject'].map(local)
    df['object_name'] = df['object'].map(local)
    df['predicate_name'] = df['predicate'].map(local)
    cols = ['stmt_id', 'subject', 'predicate', 'object',
            'subject_name', 'predicate_name', 'object_name', 'truth_value']
    return df[cols]


In [3]:
train = parse_statements(TRAIN_PATH)
test = parse_statements(TEST_PATH)

print(f"train: {train.shape}   test: {test.shape}")
print()
print("train truth_value counts:")
print(train.truth_value.value_counts(dropna=False))
train.head()


train: (1234, 8)   test: (1342, 8)

train truth_value counts:
truth_value
1.0    675
0.0    559
Name: count, dtype: int64


,stmt_id,subject,predicate,object,subject_name,predicate_name,object_name,truth_value
0,3226691,http://dbpedia.org/resource/David_Lee_(basketb...,http://dbpedia.org/ontology/team,http://dbpedia.org/resource/Houston_Rockets,David_Lee_(basketball),team,Houston_Rockets,0.0
1,3320759,http://dbpedia.org/resource/Nenad_Zimonjić,http://dbpedia.org/ontology/award,http://dbpedia.org/resource/Belgrade,Nenad_Zimonjić,award,Belgrade,0.0
2,3642843,http://dbpedia.org/resource/Om_Shanti_Om,http://dbpedia.org/ontology/starring,http://dbpedia.org/resource/Uma_Thurman,Om_Shanti_Om,starring,Uma_Thurman,0.0
3,3800661,http://dbpedia.org/resource/Walt_Whitman,http://dbpedia.org/ontology/deathPlace,"http://dbpedia.org/resource/Camden,_New_Jersey",Walt_Whitman,deathPlace,"Camden,_New_Jersey",1.0
4,3386366,http://dbpedia.org/resource/François_Jacob,http://dbpedia.org/ontology/award,http://dbpedia.org/resource/Nobel_Prize_in_Phy...,François_Jacob,award,Nobel_Prize_in_Physiology_or_Medicine,1.0


## 1. Exploratory Data Analysis


In [4]:
print("Only 9 relations occur. Per-relation breakdown (train) + counts (test):")
rel_stats = train.groupby('predicate_name')['truth_value'].agg(['count', 'mean'])
rel_stats.columns = ['n_train', 'pct_true']
rel_stats['n_test'] = test.groupby('predicate_name').size()
rel_stats.sort_values('n_train', ascending=False)


Only 9 relations occur. Per-relation breakdown (train) + counts (test):


,n_train,pct_true,n_test
predicate_name,,,
deathPlace,192,0.661458,245
birthPlace,182,0.692308,253
award,151,0.496689,149
starring,148,0.506757,146
team,146,0.513699,144
author,142,0.521127,143
foundationPlace,118,0.508475,121
spouse,105,0.409524,93
subsidiary,50,0.400000,48


In [5]:
# Entity overlap between train and test
train_subj, test_subj = set(train.subject), set(test.subject)
train_obj, test_obj = set(train.object), set(test.object)
all_ent_train = train_subj | train_obj
all_ent_test = test_subj | test_obj

print(f"distinct subjects: train={len(train_subj)} test={len(test_subj)} overlap={len(train_subj & test_subj)}")
print(f"distinct objects:  train={len(train_obj)} test={len(test_obj)} overlap={len(train_obj & test_obj)}")
print(f"distinct entities: train={len(all_ent_train)} test={len(all_ent_test)} overlap={len(all_ent_train & all_ent_test)}")

key_train = set(zip(train.subject, train.predicate))
key_test = set(zip(test.subject, test.predicate))
print(f"(subject,predicate) pairs seen in BOTH train and test: {len(key_train & key_test)}")


distinct subjects: train=702 test=734 overlap=262
distinct objects:  train=597 test=603 overlap=180
distinct entities: train=1258 test=1283 overlap=450
(subject,predicate) pairs seen in BOTH train and test: 247


In [6]:
# Repeated (subject,predicate) groups -> a classic "multiple candidate objects" pattern
grp = train.groupby(['subject', 'predicate'])
multi = grp.filter(lambda g: len(g) > 1)
print(f"(subject,predicate) pairs appearing >1x in train: "
      f"{multi[['subject','predicate']].drop_duplicates().shape[0]} out of {grp.ngroups} groups")
multi.sort_values(['subject', 'predicate'])[['subject_name', 'predicate_name', 'object_name', 'truth_value']].head(10)


(subject,predicate) pairs appearing >1x in train: 279 out of 824 groups


,subject_name,predicate_name,object_name,truth_value
7,2012_(film),starring,Oliver_Platt,1.0
104,2012_(film),starring,Art_Carney,0.0
235,2012_(film),starring,Harry_Shearer,0.0
443,2012_(film),starring,Amanda_Peet,1.0
494,2012_(film),starring,Danny_Glover,1.0
537,2012_(film),starring,Paul_McCartney,0.0
603,2012_(film),starring,Alyson_Stoner,0.0
657,2012_(film),starring,Lucy_Liu,0.0
701,2012_(film),starring,Barbara_Hershey,0.0
765,2012_(film),starring,Lucas_Grabeel,0.0


In [7]:
# Object popularity signal: some objects recur as TRUE much more than others
pop = train.groupby(['predicate_name', 'object_name'])['truth_value'].agg(['count', 'mean'])
pop = pop[pop['count'] > 1].sort_values('count', ascending=False)
pop.head(15)


count      mean
predicate_name  object_name                                           
award           Nobel_Prize_in_Literature                 82  0.573171
                Nobel_Prize_in_Physics                    29  0.448276
birthPlace      New_York_City                              9  0.555556
award           Nobel_Peace_Prize                          9  0.444444
foundationPlace California                                 9  0.666667
deathPlace      New_York_City                              9  1.000000
team            Houston_Rockets                            8  0.625000
birthPlace      London                                     7  0.285714
deathPlace      Los_Angeles                                7  1.000000
team            New_York_Knicks                            7  0.285714
                Phoenix_Suns                               7  0.857143
                Oklahoma_City_Thunder                      7  0.714286
                Dallas_Mavericks                           7  0.571429
award           Nobel_Prize_in_Physiology_or_Medicine      7  0.428571
foundationPlace United_States                              7  0.571429

In [8]:
# Exact triple overlap between train's TRUE facts and test (direct evidence for those test rows)
true_triples = set(zip(train.loc[train.truth_value == 1, 'subject'],
                        train.loc[train.truth_value == 1, 'predicate'],
                        train.loc[train.truth_value == 1, 'object']))
test_triples = set(zip(test.subject, test.predicate, test.object))
print("exact (train-true) <-> test triple overlap:", len(true_triples & test_triples))


exact (train-true) <-> test triple overlap: 35


**Findings that drive the whole approach:**
1. Repeated `(subject, predicate)` groups (multiple candidate objects, some true some false) — a classic negative-generation pattern.
2. Substantial entity overlap between train and test — the train labels are real, usable evidence for many test facts.
3. Object/subject popularity is informative — hub objects that recur as *true* many times (famous cities, Nobel categories, ...) are strong evidence of a genuine fact; one-off obscure objects tend to be corruptions.

No live DBpedia/SPARQL access is available in this environment, so the
**train+test files themselves become the knowledge base** — that's the
strategy used below, rather than querying an external KG.


## 2. Using the knowledge base — feature engineering

Two layers of evidence, kept carefully separate to avoid leakage:

- **`Pool`** — structural, *label-free* features computed from the mere
  existence of triples across the whole train+test file (safe to use in
  full: this is the standard *transductive* setting in KG-embedding
  research — the graph structure is known upfront, only the labels of the
  evaluation triples are hidden).
- **`LabelContext`** — features computed only from a set of
  *known-labelled* triples (a training fold). Uses `Counter` (not `set`)
  so a row's own label can be subtracted out (`exclude_self=True`) when
  scoring the very rows a context was built from — proper leave-one-out,
  which matters because **24 exact triples repeat within train itself**.


In [9]:
class Pool:
    """Structural statistics from the union of all (train+test) triples,
    ignoring truth labels entirely."""
    def __init__(self, all_df):
        self.subject_freq = all_df['subject'].value_counts().to_dict()
        self.object_freq = all_df['object'].value_counts().to_dict()
        self.pred_obj_freq = all_df.groupby(['predicate', 'object']).size().to_dict()
        self.subj_pred_freq = all_df.groupby(['subject', 'predicate']).size().to_dict()
        entities = pd.unique(pd.concat([all_df['subject'], all_df['object']]))
        self.entity2idx = {e: i for i, e in enumerate(entities)}
        predicates = pd.unique(all_df['predicate'])
        self.pred2idx = {p: i for i, p in enumerate(predicates)}

    def featurize(self, df):
        out = pd.DataFrame(index=df.index)
        out['subject_freq'] = df['subject'].map(self.subject_freq).fillna(0)
        out['object_freq'] = df['object'].map(self.object_freq).fillna(0)
        out['pred_obj_freq'] = [self.pred_obj_freq.get((p, o), 0)
                                 for p, o in zip(df['predicate'], df['object'])]
        out['subj_pred_freq'] = [self.subj_pred_freq.get((s, p), 0)
                                  for s, p in zip(df['subject'], df['predicate'])]
        out['log_subject_freq'] = np.log1p(out['subject_freq'])
        out['log_object_freq'] = np.log1p(out['object_freq'])
        out['log_pred_obj_freq'] = np.log1p(out['pred_obj_freq'])
        return out


In [10]:
class LabelContext:
    """Statistics derived ONLY from a set of known-labelled triples
    (e.g. a training fold). Must be refit per-fold during CV."""

    def __init__(self, labelled_df):
        self.df = labelled_df
        true_df = labelled_df[labelled_df.truth_value == 1]
        false_df = labelled_df[labelled_df.truth_value == 0]

        self.exact_true_ctr = Counter(zip(true_df.subject, true_df.predicate, true_df.object))
        self.exact_false_ctr = Counter(zip(false_df.subject, false_df.predicate, false_df.object))

        self.sp_true_objs = defaultdict(Counter)
        for s, p, o in zip(true_df.subject, true_df.predicate, true_df.object):
            self.sp_true_objs[(s, p)][o] += 1
        self.sp_false_objs = defaultdict(Counter)
        for s, p, o in zip(false_df.subject, false_df.predicate, false_df.object):
            self.sp_false_objs[(s, p)][o] += 1

        self.po_true_ctr = Counter(zip(true_df.predicate, true_df.object))
        self.po_false_ctr = Counter(zip(false_df.predicate, false_df.object))

        rel_stats = labelled_df.groupby('predicate')['truth_value'].agg(['mean', 'count'])
        self.pred_prior = rel_stats['mean'].to_dict()
        self.pred_n = rel_stats['count'].to_dict()
        self.global_prior = labelled_df['truth_value'].mean()

        g_subj = labelled_df.groupby('subject')['truth_value'].agg(['sum', 'count'])
        self.subject_true_sum = g_subj['sum'].to_dict()
        self.subject_n = g_subj['count'].to_dict()
        g_obj = labelled_df.groupby('object')['truth_value'].agg(['sum', 'count'])
        self.object_true_sum = g_obj['sum'].to_dict()
        self.object_n = g_obj['count'].to_dict()

    def featurize(self, df, exclude_self=False):
        rows = []
        it = zip(df.index, df['subject'], df['predicate'], df['object'],
                  df['truth_value'] if 'truth_value' in df.columns else [None] * len(df))
        for idx, s, p, o, y_self in it:
            key = (s, p, o)
            self_is_true = exclude_self and (y_self == 1)
            self_is_false = exclude_self and (y_self == 0)

            po_true = self.po_true_ctr.get((p, o), 0) - (1 if self_is_true else 0)
            po_false = self.po_false_ctr.get((p, o), 0) - (1 if self_is_false else 0)
            po_true = max(po_true, 0); po_false = max(po_false, 0)

            exact_true = self.exact_true_ctr.get(key, 0) - (1 if self_is_true else 0)
            exact_false = self.exact_false_ctr.get(key, 0) - (1 if self_is_false else 0)
            exact_true = max(exact_true, 0); exact_false = max(exact_false, 0)

            true_objs = self.sp_true_objs.get((s, p), Counter()).copy()
            false_objs = self.sp_false_objs.get((s, p), Counter()).copy()
            if self_is_true and true_objs.get(o, 0) > 0:
                true_objs[o] -= 1
            if self_is_false and false_objs.get(o, 0) > 0:
                false_objs[o] -= 1
            other_true_ct = sum(c for obj, c in true_objs.items() if obj != o)
            other_false_ct = sum(c for obj, c in false_objs.items() if obj != o)

            subj_sum = self.subject_true_sum.get(s, 0.0) - (1 if self_is_true else 0)
            subj_n = self.subject_n.get(s, 0) - (1 if exclude_self and y_self in (0, 1) else 0)
            obj_sum = self.object_true_sum.get(o, 0.0) - (1 if self_is_true else 0)
            obj_n = self.object_n.get(o, 0) - (1 if exclude_self and y_self in (0, 1) else 0)

            pred_prior_n = self.pred_n.get(p, 0) - (1 if exclude_self and y_self in (0, 1) else 0)
            pred_sum = self.pred_prior.get(p, self.global_prior) * self.pred_n.get(p, 0) - (1 if self_is_true else 0)
            predicate_prior = (pred_sum / pred_prior_n) if pred_prior_n > 0 else self.global_prior

            rows.append({
                'exact_true_count': exact_true,
                'exact_false_count': exact_false,
                'po_true_count': po_true,
                'po_false_count': po_false,
                'po_true_minus_false': po_true - po_false,
                'other_true_obj_count_same_sp': other_true_ct,
                'other_false_obj_count_same_sp': other_false_ct,
                'has_other_true_object_same_sp': float(other_true_ct > 0),
                'has_other_false_object_same_sp': float(other_false_ct > 0),
                'subject_true_rate': (subj_sum / subj_n) if subj_n > 0 else np.nan,
                'subject_n': max(subj_n, 0),
                'object_true_rate': (obj_sum / obj_n) if obj_n > 0 else np.nan,
                'object_n': max(obj_n, 0),
                'predicate_prior': predicate_prior,
            })
        feat = pd.DataFrame(rows, index=df.index)
        feat['subject_true_rate'] = feat['subject_true_rate'].fillna(feat['predicate_prior'])
        feat['object_true_rate'] = feat['object_true_rate'].fillna(feat['predicate_prior'])
        return feat


In [11]:
# quick demonstration
all_df = pd.concat([train, test], ignore_index=True)
pool = Pool(all_df)
label_ctx = LabelContext(train)

print("Pool features (first 5 rows):")
display(pool.featurize(train).head())
print("\nLabelContext features, leave-one-out on train itself (first 5 rows):")
display(label_ctx.featurize(train, exclude_self=True).head())


,subject_freq,object_freq,pred_obj_freq,subj_pred_freq,log_subject_freq,log_object_freq,log_pred_obj_freq
0,3,13,13,3,1.386294,2.639057,2.639057
1,2,8,1,1,1.098612,2.197225,0.693147
2,10,2,2,10,2.397895,1.098612,1.098612
3,2,1,1,2,1.098612,0.693147,0.693147
4,5,18,17,5,1.791759,2.944439,2.890372


,exact_true_count,exact_false_count,po_true_count,po_false_count,po_true_minus_false,other_true_obj_count_same_sp,other_false_obj_count_same_sp,has_other_true_object_same_sp,has_other_false_object_same_sp,subject_true_rate,subject_n,object_true_rate,object_n,predicate_prior
0,0,0,5,2,3,0,0,0.0,0.0,0.517241,0,0.714286,7,0.517241
1,0,0,0,0,0,0,0,0.0,0.0,1.000000,1,1.000000,3,0.500000
2,0,0,1,0,1,3,2,1.0,1.0,0.600000,5,1.000000,1,0.510204
3,0,0,0,0,0,0,0,0.0,0.0,0.659686,0,0.659686,0,0.659686
4,0,0,2,4,-2,0,3,0.0,1.0,0.000000,3,0.285714,7,0.493333


Pool features (first 5 rows):

LabelContext features, leave-one-out on train itself (first 5 rows):


### Optional: a from-scratch KG-embedding model (TransE-style)

`score(s,p,o) = b_p - ||e_s + r_p - e_o||` (higher = more plausible),
trained with a **logistic loss directly against the real true/false
labels** — since we already have genuine negatives, there's no need for
the usual synthetic-corruption trick.

**A bug worth recording:** the first version omitted the per-relation bias
`b_p`. Since a norm can never be negative, `score = -||d||` is always `≤ 0`,
which caps `sigmoid(score) ≤ 0.5` for *every* fact — the model could never
confidently predict "true", no matter how well it separated the classes.
Adding a learnable bias/margin term fixed it. The cell below reproduces
both versions on synthetic data so the fix is visible.


In [12]:
class TransEClassifier:
    def __init__(self, dim=32, lr=0.2, epochs=500, l2=1e-3, seed=0):
        self.dim = dim; self.lr = lr; self.epochs = epochs; self.l2 = l2; self.seed = seed

    def fit(self, n_entities, n_relations, s_idx, p_idx, o_idx, y, use_bias=True, verbose=False):
        rng = np.random.default_rng(self.seed)
        d = self.dim
        self.E = rng.normal(0, 0.5, size=(n_entities, d))
        self.R = rng.normal(0, 0.1, size=(n_relations, d))
        self.b = np.zeros(n_relations)
        self.use_bias = use_bias
        s_idx = np.asarray(s_idx); p_idx = np.asarray(p_idx)
        o_idx = np.asarray(o_idx); y = np.asarray(y, dtype=float)
        n = len(y)

        for epoch in range(self.epochs):
            d_vec = self.E[s_idx] + self.R[p_idx] - self.E[o_idx]
            norm = np.linalg.norm(d_vec, axis=1) + 1e-9
            score = (self.b[p_idx] if use_bias else 0.0) - norm
            p = 1 / (1 + np.exp(-score))

            g_score = p - y                          # dL/dscore (BCE + sigmoid identity)
            g_norm = -g_score
            grad_d = (g_norm / norm)[:, None] * d_vec

            gE = np.zeros_like(self.E); gR = np.zeros_like(self.R); gb = np.zeros_like(self.b)
            np.add.at(gE, s_idx, grad_d)
            np.add.at(gE, o_idx, -grad_d)
            np.add.at(gR, p_idx, grad_d)
            np.add.at(gb, p_idx, g_score)

            self.E -= self.lr * (gE / n + self.l2 * self.E)
            self.R -= self.lr * (gR / n + self.l2 * self.R)
            if use_bias:
                self.b -= self.lr * (gb / n)
        return self

    def score(self, s_idx, p_idx, o_idx):
        s_idx = np.asarray(s_idx); p_idx = np.asarray(p_idx); o_idx = np.asarray(o_idx)
        d_vec = self.E[s_idx] + self.R[p_idx] - self.E[o_idx]
        norm = np.linalg.norm(d_vec, axis=1)
        return (self.b[p_idx] if self.use_bias else 0.0) - norm

    def predict_proba(self, s_idx, p_idx, o_idx):
        return 1 / (1 + np.exp(-self.score(s_idx, p_idx, o_idx)))


In [13]:
# Sanity check on synthetic data matching TransE's geometric assumption
rng = np.random.default_rng(0)
n_e, n_r, d = 30, 3, 8
E_true = rng.normal(0, 1, size=(n_e, d))
R_true = rng.normal(0, 0.3, size=(n_r, d))
N = 800
s = rng.integers(0, n_e, N); p = rng.integers(0, n_r, N); o = rng.integers(0, n_e, N)
dist = np.linalg.norm(E_true[s] + R_true[p] - E_true[o], axis=1)
y = (dist < np.median(dist)).astype(float)

m_nobias = TransEClassifier(dim=d, epochs=400, lr=0.2).fit(n_e, n_r, s, p, o, y, use_bias=False)
acc_nobias = ((m_nobias.predict_proba(s, p, o) > 0.5).astype(float) == y).mean()

m_bias = TransEClassifier(dim=d, epochs=400, lr=0.2).fit(n_e, n_r, s, p, o, y, use_bias=True)
acc_bias = ((m_bias.predict_proba(s, p, o) > 0.5).astype(float) == y).mean()

print(f"WITHOUT bias term: accuracy = {acc_nobias:.3f}  (stuck at chance -- can never predict 'true' confidently)")
print(f"WITH    bias term: accuracy = {acc_bias:.3f}  (fix works)")


WITHOUT bias term: accuracy = 0.500  (stuck at chance -- can never predict 'true' confidently)
WITH    bias term: accuracy = 0.892  (fix works)


## 3. The Fact-Checking Engine

Combine: **Pool features** + **LabelContext features** + (optional)
**KGE plausibility score** + predicate one-hot &rarr; a **meta-classifier**
(Logistic Regression, isotonic-calibrated) &rarr; final veracity score.

> Spoiler (see the ablation study below): the KGE score is included here
> for completeness but is **off by default** — on this dataset size it adds
> noise rather than signal.


In [14]:
PREDICATES = ['deathPlace', 'birthPlace', 'award', 'starring', 'team',
              'author', 'foundationPlace', 'spouse', 'subsidiary']

def _predicate_local(p):
    return p.rsplit('/', 1)[-1]

def _onehot_predicate(df):
    local = df['predicate'].map(_predicate_local)
    out = pd.DataFrame(index=df.index)
    for pr in PREDICATES:
        out[f'pred_{pr}'] = (local == pr).astype(float)
    return out

def train_kge(fold_df, pool, **kge_kwargs):
    s_idx = fold_df['subject'].map(pool.entity2idx).values
    p_idx = fold_df['predicate'].map(pool.pred2idx).values
    o_idx = fold_df['object'].map(pool.entity2idx).values
    y = fold_df['truth_value'].astype(float).values
    model = TransEClassifier(**kge_kwargs)
    model.fit(len(pool.entity2idx), len(pool.pred2idx), s_idx, p_idx, o_idx, y)
    return model

def kge_features(df, pool, kge_model):
    s_idx = df['subject'].map(pool.entity2idx).values
    p_idx = df['predicate'].map(pool.pred2idx).values
    o_idx = df['object'].map(pool.entity2idx).values
    score = kge_model.score(s_idx, p_idx, o_idx)
    proba = kge_model.predict_proba(s_idx, p_idx, o_idx)
    return pd.DataFrame({'kge_score': score, 'kge_proba': proba}, index=df.index)

def build_features(df, pool, label_ctx, kge_model, exclude_self=False, use_kge=True):
    parts = [pool.featurize(df), label_ctx.featurize(df, exclude_self=exclude_self), _onehot_predicate(df)]
    if use_kge and kge_model is not None:
        parts.insert(2, kge_features(df, pool, kge_model))
    return pd.concat(parts, axis=1)


class FactCheckingEngine:
    """
    use_kge=False is the empirically-chosen default (see the ablation study):
    a TransE-style score doesn't help on ~1,200 labelled triples spread over
    ~2,000 mostly-singleton entities. Kept as a toggle for larger/denser KGs.
    """
    def __init__(self, model_type='logreg', use_kge=False, kge_dim=32,
                 kge_epochs=500, kge_lr=0.2, kge_l2=1e-3, seed=0):
        self.model_type = model_type
        self.use_kge = use_kge
        self.kge_kwargs = dict(dim=kge_dim, epochs=kge_epochs, lr=kge_lr, l2=kge_l2, seed=seed)
        self.seed = seed

    def _make_classifier(self):
        if self.model_type == 'logreg':
            return LogisticRegression(max_iter=2000, C=1.0)
        elif self.model_type == 'gboost':
            return HistGradientBoostingClassifier(max_depth=4, max_iter=150,
                                                   learning_rate=0.08, random_state=self.seed)
        raise ValueError(self.model_type)

    def fit(self, train_df, pool):
        self.pool = pool
        self.label_ctx = LabelContext(train_df)
        self.kge_model = train_kge(train_df, pool, **self.kge_kwargs) if self.use_kge else None

        X = build_features(train_df, pool, self.label_ctx, self.kge_model,
                            exclude_self=True, use_kge=self.use_kge)
        y = train_df['truth_value'].astype(float).values
        self.feature_names_ = X.columns.tolist()

        self.scaler = StandardScaler()
        Xs = self.scaler.fit_transform(X.values)

        base = self._make_classifier()
        self.clf = CalibratedClassifierCV(base, method='isotonic', cv=5)
        self.clf.fit(Xs, y)
        return self

    def predict_veracity(self, df):
        X = build_features(df, self.pool, self.label_ctx, self.kge_model,
                            exclude_self=False, use_kge=self.use_kge)
        X = X[self.feature_names_]
        Xs = self.scaler.transform(X.values)
        return self.clf.predict_proba(Xs)[:, 1]


### Baselines


In [15]:
def metrics_report(y_true, p_pred, name=''):
    y_hat = (p_pred >= 0.5).astype(int)
    return {
        'model': name,
        'accuracy': accuracy_score(y_true, y_hat),
        'precision': precision_score(y_true, y_hat, zero_division=0),
        'recall': recall_score(y_true, y_hat, zero_division=0),
        'f1': f1_score(y_true, y_hat, zero_division=0),
        'roc_auc': roc_auc_score(y_true, p_pred),
        'brier': brier_score_loss(y_true, p_pred),
        'log_loss': log_loss(y_true, p_pred, labels=[0, 1]),
    }

def baseline_majority(train):
    p = np.full(len(train), train['truth_value'].mean())
    return metrics_report(train['truth_value'].astype(int).values, p, name='majority/prior baseline')

def baseline_predicate_prior(train, n_splits=5, seed=0):
    strat_key = train['predicate'] + '__' + train['truth_value'].astype(int).astype(str)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.zeros(len(train))
    for tr_idx, va_idx in skf.split(train, strat_key):
        prior = train.iloc[tr_idx].groupby('predicate')['truth_value'].mean()
        oof[va_idx] = train.iloc[va_idx]['predicate'].map(prior).fillna(train.iloc[tr_idx]['truth_value'].mean()).values
    return oof, metrics_report(train['truth_value'].astype(int).values, oof, name='predicate-prior baseline (OOF)')

print(pd.Series(baseline_majority(train)))
print()
_, bp = baseline_predicate_prior(train)
print(pd.Series(bp))


model        majority/prior baseline
accuracy                    0.547002
precision                   0.547002
recall                           1.0
f1                          0.707177
roc_auc                          0.5
brier                       0.247791
log_loss                    0.688722
dtype: object

model        predicate-prior baseline (OOF)
accuracy                           0.570502
precision                            0.5756
recall                             0.817778
f1                                 0.675643
roc_auc                            0.597299
brier                              0.239363
log_loss                           0.671432
dtype: object


### 5-fold cross-validation benchmark

Proper leakage-free CV: `LabelContext` and the (optional) KGE model are refit **per fold**, using only that fold's training portion.


In [16]:
def cross_validate(train, pool, model_type='logreg', use_kge=False, n_splits=5, seed=0):
    strat_key = train['predicate'] + '__' + train['truth_value'].astype(int).astype(str)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof_pred = np.zeros(len(train))
    fold_metrics = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(train, strat_key)):
        tr_df = train.iloc[tr_idx].reset_index(drop=True)
        va_df = train.iloc[va_idx].reset_index(drop=True)

        eng = FactCheckingEngine(model_type=model_type, use_kge=use_kge, seed=seed)
        eng.fit(tr_df, pool)
        p_val = eng.predict_veracity(va_df)
        oof_pred[va_idx] = p_val

        m = metrics_report(va_df['truth_value'].astype(int).values, p_val, name=f'{model_type} fold{fold}')
        fold_metrics.append(m)

    overall = metrics_report(train['truth_value'].astype(int).values, oof_pred,
                              name=f'{model_type} (5-fold OOF overall)')
    return oof_pred, pd.DataFrame(fold_metrics), overall

oof_logreg, folds_logreg, overall_logreg = cross_validate(train, pool, model_type='logreg', use_kge=False)
print(folds_logreg[['model', 'accuracy', 'roc_auc', 'f1', 'brier']])
print()
print(pd.Series(overall_logreg))


          model  accuracy   roc_auc        f1     brier
0  logreg fold0  0.821862  0.909380  0.843972  0.122406
1  logreg fold1  0.862348  0.927034  0.879433  0.102542
2  logreg fold2  0.829960  0.897189  0.851064  0.122868
3  logreg fold3  0.838057  0.905197  0.863014  0.118906
4  logreg fold4  0.825203  0.917877  0.841328  0.113725

model        logreg (5-fold OOF overall)
accuracy                        0.835494
precision                       0.821526
recall                          0.893333
f1                              0.855926
roc_auc                         0.911726
brier                           0.116091
log_loss                        0.393976
dtype: object


### Ablation study — which evidence actually matters


In [17]:
BLOCKS = {
    'predicate_onehot_only': lambda pool, lc, kge, df, excl: [_onehot_predicate(df)],
    'pool_only': lambda pool, lc, kge, df, excl: [pool.featurize(df), _onehot_predicate(df)],
    'label_only': lambda pool, lc, kge, df, excl: [lc.featurize(df, exclude_self=excl), _onehot_predicate(df)],
    'pool_plus_label (no KGE)': lambda pool, lc, kge, df, excl: [pool.featurize(df), lc.featurize(df, exclude_self=excl), _onehot_predicate(df)],
    'kge_only': lambda pool, lc, kge, df, excl: [kge_features(df, pool, kge), _onehot_predicate(df)],
    'full (pool+label+kge)': lambda pool, lc, kge, df, excl: [pool.featurize(df), lc.featurize(df, exclude_self=excl), kge_features(df, pool, kge), _onehot_predicate(df)],
}

def run_block(block_fn, train, pool, n_splits=5, seed=0):
    strat_key = train['predicate'] + '__' + train['truth_value'].astype(int).astype(str)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.zeros(len(train))
    for tr_idx, va_idx in skf.split(train, strat_key):
        tr_df = train.iloc[tr_idx].reset_index(drop=True)
        va_df = train.iloc[va_idx].reset_index(drop=True)
        lc = LabelContext(tr_df)
        kge = train_kge(tr_df, pool, dim=32, epochs=500, lr=0.2, l2=1e-3, seed=seed)

        Xtr = pd.concat(block_fn(pool, lc, kge, tr_df, True), axis=1)
        Xva = pd.concat(block_fn(pool, lc, kge, va_df, False), axis=1)
        Xva = Xva[Xtr.columns]

        scaler = StandardScaler()
        Xtr_s = scaler.fit_transform(Xtr.values)
        Xva_s = scaler.transform(Xva.values)

        clf = CalibratedClassifierCV(LogisticRegression(max_iter=2000), method='isotonic', cv=5)
        clf.fit(Xtr_s, tr_df['truth_value'].astype(float).values)
        oof[va_idx] = clf.predict_proba(Xva_s)[:, 1]
    return oof, metrics_report(train['truth_value'].astype(int).values, oof)

ablation_results = []
for name, fn in BLOCKS.items():
    _, m = run_block(fn, train, pool)
    m['model'] = name
    ablation_results.append(m)

ablation_df = pd.DataFrame(ablation_results)[['model', 'accuracy', 'roc_auc', 'f1', 'brier']]
ablation_df


,model,accuracy,roc_auc,f1,brier
0,predicate_onehot_only,0.570502,0.594485,0.697834,0.239385
1,pool_only,0.666937,0.744971,0.682135,0.199642
2,label_only,0.777958,0.864996,0.803725,0.149368
3,pool_plus_label (no KGE),0.835494,0.911726,0.855926,0.116091
4,kge_only,0.547812,0.563122,0.568779,0.261366
5,full (pool+label+kge),0.820908,0.893694,0.839273,0.130546


### Per-relation breakdown (final engine, out-of-fold)


In [18]:
tmp = train.copy()
tmp['pred_score'] = oof_logreg
tmp['pred_label'] = (tmp['pred_score'] >= 0.5).astype(int)
per_rel = tmp.groupby('predicate_name').apply(lambda g: pd.Series({
    'n': len(g),
    'pct_true': g['truth_value'].mean(),
    'accuracy': accuracy_score(g['truth_value'].astype(int), g['pred_label']),
    'roc_auc': roc_auc_score(g['truth_value'].astype(int), g['pred_score']) if g['truth_value'].nunique() > 1 else np.nan,
}), include_groups=False)
per_rel.sort_values('n', ascending=False)


,n,pct_true,accuracy,roc_auc
predicate_name,,,,
deathPlace,192.0,0.661458,0.932292,0.967717
birthPlace,182.0,0.692308,0.923077,0.959751
award,151.0,0.496689,0.801325,0.918158
starring,148.0,0.506757,0.831081,0.866393
team,146.0,0.513699,0.671233,0.689014
author,142.0,0.521127,0.838028,0.950219
foundationPlace,118.0,0.508475,0.813559,0.884770
spouse,105.0,0.409524,0.809524,0.888972
subsidiary,50.0,0.400000,0.840000,0.909167


`team` is the clear weak spot — its false candidates are evidently
same-sport/near-miss teams that are hard to rule out from `(s,p,o)`
structure alone, without richer context (season/era) that isn't in this
dataset.


### Gradient boosting comparison (for rigor)

A more flexible meta-model was also tried; it consistently *underperforms* logistic regression at this sample size.


In [19]:
oof_gb, folds_gb, overall_gb = cross_validate(train, pool, model_type='gboost', use_kge=False)
print(pd.Series(overall_gb))
print()
print("--> Logistic Regression remains the shipped default (better accuracy, AUC, and calibration).")


model        gboost (5-fold OOF overall)
accuracy                        0.570502
precision                       0.601969
recall                          0.634074
f1                              0.617605
roc_auc                         0.653927
brier                           0.368329
log_loss                        8.902915
dtype: object

--> Logistic Regression remains the shipped default (better accuracy, AUC, and calibration).


## Final model: fit on full train, score every test fact

`Pool` is rebuilt from **train+test** (structural features only, no
labels), then the engine is fit on the full labelled training set and used
to score all 1,342 test facts.


In [20]:
all_df_full = pd.concat([train, test], ignore_index=True)
pool_full = Pool(all_df_full)

final_engine = FactCheckingEngine(model_type='logreg', use_kge=False, seed=0)
final_engine.fit(train, pool_full)

veracity = final_engine.predict_veracity(test)
test_predictions = test[['stmt_id', 'subject_name', 'predicate_name', 'object_name']].copy()
test_predictions['veracity_score'] = veracity
test_predictions['predicted_label'] = np.where(test_predictions['veracity_score'] >= 0.5, 'true', 'false')
test_predictions = test_predictions.sort_values('veracity_score', ascending=False).reset_index(drop=True)

import os
_out_dir = os.path.join('..', 'outputs')
os.makedirs(_out_dir, exist_ok=True)
_out_path = os.path.join(_out_dir, 'test_predictions.csv')
test_predictions.to_csv(_out_path, index=False)
print(f"wrote {len(test_predictions)} predictions to {_out_path}")
print()
print("veracity score distribution:")
print(test_predictions['veracity_score'].describe())
print()
print("predicted-true rate on test:", (test_predictions['predicted_label'] == 'true').mean())
test_predictions.head(10)


wrote 1342 predictions to ../outputs/test_predictions.csv

veracity score distribution:
count    1342.000000
mean        0.546851
std         0.330244
min         0.000000
25%         0.327526
50%         0.599365
75%         0.794098
max         1.000000
Name: veracity_score, dtype: float64

predicted-true rate on test: 0.6974664679582713


,stmt_id,subject_name,predicate_name,object_name,veracity_score,predicted_label
0,3623252,The_Aristocats,starring,Sterling_Holloway,1.0,true
1,3871979,Sōsuke_Uno,birthPlace,"Sevier_County,_Tennessee",1.0,true
2,3614659,A_Hard_Day's_Night_(film),starring,The_Beatles,1.0,true
3,3861976,Oracle_Corporation,subsidiary,List_of_acquisitions_by_Oracle,1.0,true
4,3874373,Simeon_Saxe-Coburg-Gotha,birthPlace,"Sevier_County,_Tennessee",1.0,true
5,3615976,A_Hard_Day's_Night_(film),starring,The_Beatles,1.0,true
6,3639558,Camp_Rock,starring,Demi_Lovato,1.0,true
7,3689099,A_Hard_Day's_Night_(film),starring,The_Beatles,1.0,true
8,3670014,Mad_Max,starring,Mel_Gibson,1.0,true
9,3829893,Oracle_Corporation,subsidiary,List_of_acquisitions_by_Oracle,1.0,true


**Note:** 69.7% of test facts score &ge; 0.5 ("true"), noticeably higher
than train's 54.7% true rate. Part of this is explained by test containing
relatively more `birthPlace`/`deathPlace` facts (the highest-true-rate
relations), and part by the legitimate transductive advantage of shared
subjects/objects/exact-triples between train and test. Since test has no
gold labels here, this gap is flagged rather than hidden — the first thing
worth checking if gold labels ever become available.


## 4. Interactive single-fact checking

`FactChecker.check(subject, predicate, object)` accepts bare DBpedia local
names or full URIs, and returns the veracity score plus human-readable
**evidence** for it. It also flags `low_confidence=True` (and shrinks the
score toward the predicate's base rate) whenever the subject/object never
appears anywhere in the known graph — honest uncertainty instead of false
certainty on truly novel entities.


In [21]:
DBPEDIA_RES = 'http://dbpedia.org/resource/'
DBPEDIA_ONT = 'http://dbpedia.org/ontology/'

def _to_entity_uri(name):
    return name if name.startswith('http://') else DBPEDIA_RES + name

def _to_predicate_uri(name):
    return name if name.startswith('http://') else DBPEDIA_ONT + name


class FactChecker:
    def __init__(self, engine, pool):
        self.engine = engine
        self.pool = pool

    def check(self, subject, predicate, obj, explain=True):
        s_uri, p_uri, o_uri = _to_entity_uri(subject), _to_predicate_uri(predicate), _to_entity_uri(obj)
        row = pd.DataFrame([{'subject': s_uri, 'predicate': p_uri, 'object': o_uri}])
        veracity = float(self.engine.predict_veracity(row)[0])

        subj_known = s_uri in self.pool.entity2idx
        obj_known = o_uri in self.pool.entity2idx
        low_confidence = not (subj_known and obj_known)
        if low_confidence:
            prior = self.engine.label_ctx.pred_prior.get(p_uri, self.engine.label_ctx.global_prior)
            veracity = 0.5 * veracity + 0.5 * prior

        result = {'subject': subject, 'predicate': predicate, 'object': obj,
                  'veracity': round(veracity, 4), 'low_confidence': low_confidence}

        if explain:
            lc = self.engine.label_ctx
            key = (s_uri, p_uri, o_uri)
            evidence = []
            if lc.exact_true_ctr.get(key, 0) > 0:
                evidence.append("this exact fact already appears as TRUE in the training KG")
            if lc.exact_false_ctr.get(key, 0) > 0:
                evidence.append("this exact fact already appears as FALSE in the training KG")
            other_true = [x for x in lc.sp_true_objs.get((s_uri, p_uri), {}) if x != o_uri]
            if other_true:
                evidence.append(f"a different, already-confirmed-true object exists for "
                                 f"({subject}, {predicate}): {[x.rsplit('/',1)[-1] for x in other_true][:3]}")
            po_true = lc.po_true_ctr.get((p_uri, o_uri), 0)
            if po_true > 0:
                evidence.append(f"'{obj}' is already a confirmed-true object of '{predicate}' "
                                 f"for {po_true} other subject(s)")
            if not evidence:
                evidence.append("no direct training-KG evidence found; score is based on "
                                 "entity/predicate popularity patterns only")
            if low_confidence:
                evidence.append("LOW CONFIDENCE: subject and/or object never appear in the "
                                 f"known knowledge graph -- score blended toward the '{predicate}' base rate")
            result['evidence'] = evidence
        return result

fc = FactChecker(final_engine, pool_full)


In [22]:
examples = [
    # a true fact straight from the training KG
    ("Walt_Whitman", "deathPlace", "Camden,_New_Jersey"),
    # a plausible-looking but false fact (wrong Nobel category)
    ("François_Jacob", "award", "Nobel_Prize_in_Literature"),
    # an unseen, made-up combination (cold start)
    ("Some_Random_Person_Not_In_KG", "birthPlace", "Some_Random_City_Not_In_KG"),
]

for s, p, o in examples:
    r = fc.check(s, p, o)
    print(f"({s}, {p}, {o})")
    print(f"  veracity = {r['veracity']}")
    for e in r['evidence']:
        print(f"  - {e}")
    print()


(Walt_Whitman, deathPlace, Camden,_New_Jersey)
  veracity = 0.9891
  - this exact fact already appears as TRUE in the training KG
  - 'Camden,_New_Jersey' is already a confirmed-true object of 'deathPlace' for 1 other subject(s)

(François_Jacob, award, Nobel_Prize_in_Literature)
  veracity = 0.0
  - this exact fact already appears as FALSE in the training KG
  - a different, already-confirmed-true object exists for (François_Jacob, award): ['Nobel_Prize_in_Physiology_or_Medicine']
  - 'Nobel_Prize_in_Literature' is already a confirmed-true object of 'award' for 47 other subject(s)

(Some_Random_Person_Not_In_KG, birthPlace, Some_Random_City_Not_In_KG)
  veracity = 0.8462
  - no direct training-KG evidence found; score is based on entity/predicate popularity patterns only
  - LOW CONFIDENCE: subject and/or object never appear in the known knowledge graph -- score blended toward the 'birthPlace' base rate



### Try your own fact:


In [23]:
# fc.check("Barack_Obama", "spouse", "Michelle_Obama")


## Conclusions & limitations

- **Final engine:** Logistic Regression over Pool + LabelContext features,
  isotonic-calibrated &rarr; **83.5% accuracy / 0.91 ROC-AUC** (5-fold CV on
  train) vs. 55&ndash;57% for majority-class/predicate-prior baselines.
- The from-scratch KG-embedding model works correctly (after fixing the
  missing-bias-term bug) but the ablation study shows it **doesn't help**
  at this dataset size (~1,200 facts / ~2,000 mostly-singleton entities) —
  it's disabled by default, kept as an option for larger/denser graphs.
- Gradient boosting was tried and consistently **underperforms** logistic
  regression here — a flexible model overfitting an already
  well-featurized, small dataset.
- **`team`** is the weakest relation (67% accuracy / 0.69 AUC) — its false
  candidates are hard-to-rule-out near-miss teams from `(s,p,o)` alone.
- No live DBpedia/external KG access is available in this environment; the
  "knowledge base" is the train+test files themselves. A production
  version could add real entity types and a live SPARQL endpoint,
  especially to help with `team`.
- Benchmark numbers are a rigorous, leakage-free 5-fold CV estimate on
  train; test-set predictions can't be numerically validated here since
  gold labels for test weren't provided — the elevated predicted-true rate
  on test (69.7% vs. train's 54.7%) is flagged rather than hidden.
- Each candidate fact is scored independently; the model doesn't yet
  enforce "exactly one true object" constraints across a shared
  `(subject,predicate)` group, which the data suggests would help further.
